## Feature Maps

In [2]:
! pip install torch

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp310-cp310-win_amd64.whl.metadata (2.8 kB)
   ---------------------------------------- 0.0/111.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/111.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/111.0 MB 3.0 MB/s eta 0:00:38
   ---------------------------------------- 1.3/111.0 MB 2.6 MB/s eta 0:00:43
   ---------------------------------------- 1.3/111.0 MB 2.6 MB/s eta 0:00:43
    --------------------------------------- 1.6/111.0 MB 1.6 MB/s eta 0:01:07
    --------------------------------------- 1.6/111.0 MB 1.6 MB/s eta 0:01:07
    --------------------------------------- 1.6/111.0 MB 1.6 MB/s eta 0:01:07
    --------------------------------------- 1.6/111.0 MB 1.6 

In [1]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from PIL import Image

# Load pretrained model
model = models.vgg16(pretrained=True)
model.eval()

# Load image
img = Image.open("image.jpg").convert("RGB")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

x = transform(img).unsqueeze(0)

# Extract feature maps (first conv layer)
with torch.no_grad():
    features = model.features[0](x)

# Plot feature maps
num_filters = features.shape[1]
plt.figure(figsize=(12, 8))

for i in range(min(num_filters, 16)):
    plt.subplot(4, 4, i + 1)
    plt.imshow(features[0, i].cpu(), cmap="gray")
    plt.axis("off")

plt.suptitle("CNN Feature Maps (First Layer)")
plt.show()


ModuleNotFoundError: No module named 'torch'

### Feature Maps from Multiple CNN Layers (PyTorch)

In [3]:
layers = [0, 5, 10]  # conv layers
outputs = []

with torch.no_grad():
    x_temp = x
    for i, layer in enumerate(model.features):
        x_temp = layer(x_temp)
        if i in layers:
            outputs.append(x_temp)

# Plot
for layer_idx, fmap in zip(layers, outputs):
    plt.figure(figsize=(10, 6))
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        plt.imshow(fmap[0, i].cpu(), cmap="gray")
        plt.axis("off")
    plt.suptitle(f"Feature Maps from Layer {layer_idx}")
    plt.show()


NameError: name 'torch' is not defined

### Feature Maps using Hooks (Dynamic – PyTorch Best Practice)

In [ ]:
activation = {}

def hook_fn(module, input, output):
    activation["conv1"] = output.detach()

model.features[0].register_forward_hook(hook_fn)

_ = model(x)

fmap = activation["conv1"]

plt.figure(figsize=(10, 6))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(fmap[0, i].cpu(), cmap="gray")
    plt.axis("off")
plt.show()


### Feature Maps in Keras / TensorFlow

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

# Load model
model = tf.keras.applications.VGG16(weights="imagenet", include_top=False)

# Load image
img = tf.keras.preprocessing.image.load_img(
    "image.jpg", target_size=(224, 224)
)
img = tf.keras.preprocessing.image.img_to_array(img)
img = img / 255.0
img = img[None, ...]

# Create feature extractor
layer_outputs = [layer.output for layer in model.layers[:5]]
feature_model = tf.keras.Model(inputs=model.input, outputs=layer_outputs)

# Get feature maps
feature_maps = feature_model.predict(img)

# Plot
for fmap in feature_maps:
    plt.figure(figsize=(10, 6))
    for i in range(min(8, fmap.shape[-1])):
        plt.subplot(2, 4, i + 1)
        plt.imshow(fmap[0, :, :, i], cmap="viridis")
        plt.axis("off")
    plt.show()


### Learned Filter Visualization (Weights, Not Feature Maps)

In [ ]:
weights = model.features[0].weight.data

plt.figure(figsize=(8, 8))
for i in range(16):
    w = weights[i]
    w = (w - w.min()) / (w.max() - w.min())
    plt.subplot(4, 4, i + 1)
    plt.imshow(w.permute(1, 2, 0))
    plt.axis("off")

plt.suptitle("Learned CNN Filters")
plt.show()


### Feature Map Overlay on Image (Interpretability)

In [ ]:
import numpy as np

fmap = features[0, 0].cpu().numpy()
fmap = (fmap - fmap.min()) / (fmap.max() - fmap.min())

plt.imshow(img.resize((224, 224)))
plt.imshow(fmap, cmap="jet", alpha=0.5)
plt.axis("off")
plt.title("Feature Map Overlay")
plt.show()
